# nano-vllm: Test Notebook

Tests TurboQuant KV cache, INT8 KV cache, flash-attention backends, block manager, and scheduler.  
Requires a GPU runtime (**Runtime → Change runtime type → T4 GPU**).

## 0. Setup

In [ ]:
import os, sys

# clone repo if running in Colab 
if 'google.colab' in sys.modules or os.path.exists('/content'):
    if not os.path.exists('/content/nano-vllm'):
        !git clone https://github.com/GeeeekExplorer/nano-vllm /content/nano-vllm
    REPO = '/content/nano-vllm'
else:
    REPO = os.path.dirname(os.path.abspath('__file__')) or '.'

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print(f'Repo root: {REPO}')

In [ ]:
!pip install -q xxhash tqdm triton>=3.0.0
!pip install -q flash-attn --no-build-isolation
print('Dependencies installed.')

In [ ]:
import torch
CUDA = torch.cuda.is_available()
DEVICE = 'cuda' if CUDA else 'cpu'
print(f'CUDA: {CUDA}')
if CUDA:
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — GPU-only cells will be skipped.')

## 1. Registry Tests

In [ ]:
from nanovllm.models.registry import ModelRegistry
from nanovllm.kvcache.base import KVCacheRegistry
from nanovllm.layers.flash_attn_backend import FlashAttentionRegistry

# trigger registration side-effects
import nanovllm.kvcache           # registers default / int8 / turboquant
import nanovllm.layers.default_flash_attn
import nanovllm.layers.turboquant_flash_attn
import nanovllm.models.qwen3

print('=== Model registry ===')
print(' models:', ModelRegistry.list_models())
print(' archs :', ModelRegistry.list_architectures())
assert 'qwen3' in ModelRegistry.list_models()
assert 'Qwen3ForCausalLM' in ModelRegistry.list_architectures()

print('\n=== KV-cache registry ===')
print(' caches:', KVCacheRegistry.list_caches())
for name in ('default', 'int8', 'turboquant'):
    assert name in KVCacheRegistry.list_caches(), f'{name} not registered'

print('\n=== Flash-attn registry ===')
print(' backends:', FlashAttentionRegistry.list_backends())
for name in ('default', 'turboquant'):
    assert name in FlashAttentionRegistry.list_backends(), f'{name} not registered'

print('\nAll registry tests PASSED.')

## 2. Sequence & BlockManager (pure Python)

In [ ]:
from nanovllm.engine.sequence import Sequence
from nanovllm.sampling_params import SamplingParams

BLOCK_SIZE = 16
Sequence.block_size = BLOCK_SIZE

def make_seq(n):
    return Sequence(list(range(n)), SamplingParams(temperature=0.0, max_tokens=64))

# basic properties
for n, expected_blocks in [(1,1),(16,1),(17,2),(32,2),(33,3)]:
    s = make_seq(n)
    assert s.num_blocks == expected_blocks, f'n={n}: got {s.num_blocks}'
    lbt = n - (expected_blocks - 1) * BLOCK_SIZE
    assert s.last_block_num_tokens == lbt, f'n={n}: lbt {s.last_block_num_tokens} != {lbt}'

# append_token 
s = make_seq(5)
s.append_token(99)
assert len(s) == 6
assert s.last_token == 99
assert s.num_completion_tokens == 1

# block() slicing
s = make_seq(BLOCK_SIZE * 2 + 3)
assert len(s.block(0)) == BLOCK_SIZE
assert len(s.block(1)) == BLOCK_SIZE
assert len(s.block(2)) == 3

print('Sequence tests PASSED.')

In [ ]:
from nanovllm.engine.block_manager import BlockManager

NUM_BLOCKS = 64
BM = BlockManager(NUM_BLOCKS, BLOCK_SIZE)

# allocate & deallocate 
s = make_seq(BLOCK_SIZE * 2 + 5)   # 3 blocks
assert BM.can_allocate(s)
BM.allocate(s)
assert len(s.block_table) == 3
assert len(BM.free_block_ids) == NUM_BLOCKS - 3

# last block must be unsealed (bug-fix check)
last_blk = BM.blocks[s.block_table[-1]]
assert last_blk.hash == -1, 'Last block must NOT be sealed after allocate()'

# previous full blocks must be sealed
for idx in s.block_table[:-1]:
    assert BM.blocks[idx].hash != -1, f'Block {idx} should be sealed'

BM.deallocate(s)
assert len(BM.free_block_ids) == NUM_BLOCKS
assert s.block_table == []

print('allocate / deallocate PASSED.')

# edge case: prompt length exactly = block_size (the critical bug fix)
BM2 = BlockManager(NUM_BLOCKS, BLOCK_SIZE)
s_exact = make_seq(BLOCK_SIZE)   # exactly 1 full block
BM2.allocate(s_exact)
last = BM2.blocks[s_exact.block_table[-1]]
assert last.hash == -1, 'Must be unsealed — may_append() needs to seal it'
print('Exact-block-size edge case PASSED.')

# may_append simulation across several decode steps
BM3 = BlockManager(NUM_BLOCKS, BLOCK_SIZE)
s3 = make_seq(BLOCK_SIZE - 2)    # partial last block
BM3.allocate(s3)

for step in range(BLOCK_SIZE * 2 + 4):   # cross two block boundaries
    assert BM3.can_append(s3), f'Step {step}: cannot append'
    BM3.may_append(s3)
    s3.append_token(step)

print(f'may_append across {BLOCK_SIZE*2+4} steps PASSED.')
print(f'Final block_table length: {len(s3.block_table)}')

In [ ]:
from nanovllm.engine.scheduler import Scheduler
from nanovllm.config import Config
from unittest.mock import MagicMock

# Build a minimal fake config (no real model path needed here)
cfg = MagicMock(spec=Config)
cfg.max_num_seqs = 8
cfg.max_num_batched_tokens = 512
cfg.eos = 2
cfg.num_kvcache_blocks = 64
cfg.kvcache_block_size = BLOCK_SIZE

sched = Scheduler(cfg)

# Add 3 sequences
for i in range(3):
    sched.add(make_seq(10 + i))

seqs, is_prefill = sched.schedule()
assert is_prefill
assert len(seqs) == 3

# Fake token ids (not EOS)
token_ids = [5, 6, 7]
sched.postprocess(seqs, token_ids)
assert all(len(s) > 0 for s in seqs)

# Next schedule should be decode
seqs2, is_prefill2 = sched.schedule()
assert not is_prefill2
assert len(seqs2) == 3

print('Scheduler tests PASSED.')

## 3. KV Cache Backends (GPU)

In [ ]:
if not CUDA:
    print('No GPU — skipping KV cache tests.')
else:
    # shared params
    NUM_BLOCKS  = 32
    BLOCK_SZ    = 16
    NUM_HEADS   = 8
    HEAD_DIM    = 64   # must be power-of-2 for TurboQuant
    DTYPE       = torch.float16
    N_TOKENS    = 12

    # generate random key/value tensors
    key   = torch.randn(N_TOKENS, NUM_HEADS, HEAD_DIM, dtype=DTYPE, device='cuda')
    value = torch.randn(N_TOKENS, NUM_HEADS, HEAD_DIM, dtype=DTYPE, device='cuda')
    slot_mapping = torch.arange(N_TOKENS, dtype=torch.int32, device='cuda')

    # make key/value contiguous with correct strides
    key   = key.contiguous()
    value = value.contiguous()
    print(f'key shape : {key.shape}, stride: {key.stride()}')
    print(f'value shape: {value.shape}, stride: {value.stride()}')
    print('Shared tensors created.')

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    from nanovllm.kvcache.default import DefaultKVCache

    cache = DefaultKVCache(
        num_blocks=NUM_BLOCKS, block_size=BLOCK_SZ,
        num_heads=NUM_HEADS,   head_dim=HEAD_DIM,
        dtype=DTYPE,
    )
    k_c, v_c = cache.allocate()
    print(f'DefaultKVCache allocated: k_cache={k_c.shape}, v_cache={v_c.shape}')
    assert k_c.dtype == DTYPE
    assert k_c.shape == (NUM_BLOCKS, BLOCK_SZ, NUM_HEADS, HEAD_DIM)

    cache.store(key, value, k_c, v_c, slot_mapping)
    torch.cuda.synchronize()

    # verify a couple of slots
    for i in [0, 5, 11]:
        blk, pos = i // BLOCK_SZ, i % BLOCK_SZ
        stored_k = k_c[blk, pos]    # (NUM_HEADS, HEAD_DIM)
        assert torch.allclose(stored_k, key[i]), f'slot {i} mismatch'

    # retrieve is a no-op for default
    rk, rv = cache.retrieve(k_c, v_c)
    assert rk is k_c and rv is v_c
    assert not cache.needs_dequantization()

    print('DefaultKVCache PASSED.')

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    from nanovllm.kvcache.int8 import Int8KVCache

    i8 = Int8KVCache(
        num_blocks=NUM_BLOCKS, block_size=BLOCK_SZ,
        num_heads=NUM_HEADS,   head_dim=HEAD_DIM,
        dtype=DTYPE,
    )
    k_c8, v_c8, k_sc, v_sc = i8.allocate()
    print(f'Int8KVCache allocated:')
    print(f'  k_cache={k_c8.shape} dtype={k_c8.dtype}')
    print(f'  k_scales={k_sc.shape} dtype={k_sc.dtype}')
    assert k_c8.dtype == torch.int8
    assert k_sc.dtype == torch.float32
    assert k_sc.shape == (NUM_BLOCKS * BLOCK_SZ,)

    i8.store(key, value, k_c8, v_c8, slot_mapping, k_sc, v_sc)
    torch.cuda.synchronize()

    # scales must be positive after storing non-zero tensors
    stored_slots = k_sc[:N_TOKENS]
    assert (stored_slots > 0).all(), 'INT8 k_scales should be > 0'

    # retrieve dequantizes back to DTYPE
    rk, rv = i8.retrieve(k_c8, v_c8, k_sc, v_sc)
    assert rk.dtype == DTYPE
    assert rk.shape == (NUM_BLOCKS, BLOCK_SZ, NUM_HEADS, HEAD_DIM)
    assert i8.needs_dequantization()

    # quantise-dequantise round-trip should be close (not exact)
    for i in [0, 5, 11]:
        blk, pos = i // BLOCK_SZ, i % BLOCK_SZ
        orig = key[i].float()
        recovered = rk[blk, pos].float()
        rel_err = (orig - recovered).abs().mean() / orig.abs().mean().clamp(min=1e-6)
        assert rel_err < 0.05, f'slot {i}: INT8 rel error {rel_err:.4f} too large'

    print('Int8KVCache PASSED (mean round-trip error < 5%).')

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    from nanovllm.kvcache.turboquant import TurboQuantKVCache

    tq = TurboQuantKVCache(
        num_blocks=NUM_BLOCKS, block_size=BLOCK_SZ,
        num_heads=NUM_HEADS,   head_dim=HEAD_DIM,
        dtype=DTYPE,
    )

    tensors = tq.allocate()
    k_cq, v_cq, k_norms, v_scales, v_zeros, k_centroids, rotation = tensors
    print('TurboQuantKVCache allocated:')
    print(f'  k_cache   {k_cq.shape}  dtype={k_cq.dtype}   (packed uint8)')
    print(f'  v_cache   {v_cq.shape}  dtype={v_cq.dtype}')
    print(f'  k_norms   {k_norms.shape}  dtype={k_norms.dtype}')
    print(f'  v_scales  {v_scales.shape}  dtype={v_scales.dtype}')
    print(f'  v_zeros   {v_zeros.shape}  dtype={v_zeros.dtype}')
    print(f'  k_centroids {k_centroids.shape}')
    print(f'  rotation  {rotation.shape}')

    assert k_cq.dtype == torch.uint8
    assert k_cq.shape == (NUM_BLOCKS, BLOCK_SZ, NUM_HEADS, tq.key_packed_bytes)
    assert len(tensors) == 7

    # store — runs the Triton quantisation kernel
    tq.store(key, value, k_cq, v_cq, slot_mapping,
             k_norms, v_scales, v_zeros, k_centroids, rotation)
    torch.cuda.synchronize()

    # k_norms for used slots should be > 0 (L2 norm of non-zero keys)
    used_norms = k_norms.view(-1)[:N_TOKENS]
    assert (used_norms > 0).all(), 'k_norms must be positive for non-zero keys'

    # v_scales for used slots should be > 0
    used_scales = v_scales.view(-1)[:N_TOKENS]
    assert (used_scales > 0).all(), 'v_scales must be positive'

    # packed bytes: head_dim=64, 4-bit → 32 bytes/head
    assert tq.key_packed_bytes == HEAD_DIM // 2

    # retrieve raises NotImplementedError (fused kernel not yet implemented)
    try:
        tq.retrieve(k_cq, v_cq, k_norms, v_scales, v_zeros, k_centroids, rotation)
        print('WARNING: expected NotImplementedError from retrieve()')
    except NotImplementedError:
        print('retrieve() correctly raises NotImplementedError.')

    print('TurboQuantKVCache store kernel PASSED.')

## 4. Flash Attention Backends (GPU)

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    # Shared attention tensors — variable-length prefill
    # Batch: 2 sequences of length 10 and 14
    SEQ_LENS = [10, 14]
    TOTAL    = sum(SEQ_LENS)
    NH, NKV, HD = 8, 4, 64    # GQA: 8 query heads, 4 kv heads
    SCALE    = HEAD_DIM ** -0.5

    import torch
    q_pf = torch.randn(TOTAL, NH,  HD, dtype=DTYPE, device='cuda')
    k_pf = torch.randn(TOTAL, NKV, HD, dtype=DTYPE, device='cuda')
    v_pf = torch.randn(TOTAL, NKV, HD, dtype=DTYPE, device='cuda')

    cu_q = torch.tensor([0, SEQ_LENS[0], TOTAL], dtype=torch.int32, device='cuda')
    cu_k = cu_q.clone()
    max_q = max(SEQ_LENS)
    max_k = max_q

    print(f'Prefill tensors: q={q_pf.shape}, k={k_pf.shape}')

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    from nanovllm.layers.default_flash_attn import DefaultFlashAttention

    dfa = DefaultFlashAttention()

    # prefill
    out_pf = dfa.prefill(
        q_pf, k_pf, v_pf,
        scale=SCALE,
        max_seqlen_q=max_q, cu_seqlens_q=cu_q,
        max_seqlen_k=max_k, cu_seqlens_k=cu_k,
    )
    assert out_pf.shape == (TOTAL, NH, HD), f'prefill output shape wrong: {out_pf.shape}'
    assert out_pf.dtype == DTYPE
    assert not out_pf.isnan().any(), 'NaN in prefill output'
    print(f'DefaultFlashAttention prefill output: {out_pf.shape}  PASSED.')

    # decode: batch of 4, reading from paged kv-cache
    BS_DEC = 4
    NB_DEC = 8     # number of cache blocks
    BK_DEC = 16
    q_dc = torch.randn(BS_DEC, 1, NH, HD, dtype=DTYPE, device='cuda')
    kc_dc = torch.randn(NB_DEC, BK_DEC, NKV, HD, dtype=DTYPE, device='cuda')
    vc_dc = torch.randn(NB_DEC, BK_DEC, NKV, HD, dtype=DTYPE, device='cuda')
    cache_seqlens = torch.tensor([5, 8, 3, 12], dtype=torch.int32, device='cuda')
    # simple block table: seq i uses block i
    block_tables = torch.zeros(BS_DEC, NB_DEC, dtype=torch.int32, device='cuda')
    for i in range(BS_DEC):
        block_tables[i, 0] = i

    out_dc = dfa.decode(
        q_dc, kc_dc, vc_dc,
        scale=SCALE,
        cache_seqlens=cache_seqlens,
        block_table=block_tables,
    )
    # flash_attn_with_kvcache returns (bs, seqlen_q, nheads, headdim)
    assert out_dc.shape == (BS_DEC, 1, NH, HD), f'decode output shape wrong: {out_dc.shape}'
    assert not out_dc.isnan().any(), 'NaN in decode output'
    print(f'DefaultFlashAttention decode  output: {out_dc.shape}  PASSED.')

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    from nanovllm.layers.turboquant_flash_attn import TurboQuantFlashAttention

    tqfa = TurboQuantFlashAttention()

    # prefill: uses full-precision k/v (same as DefaultFlashAttention)
    out_tq_pf = tqfa.prefill(
        q_pf, k_pf, v_pf,
        scale=SCALE,
        max_seqlen_q=max_q, cu_seqlens_q=cu_q,
        max_seqlen_k=max_k, cu_seqlens_k=cu_k,
    )
    assert out_tq_pf.shape == (TOTAL, NH, HD)
    assert not out_tq_pf.isnan().any()

    # TurboQuant prefill output should match Default (same call, no quantisation during prefill)
    assert torch.allclose(out_tq_pf, out_pf, atol=1e-3), \
        'TurboQuant and Default prefill outputs differ unexpectedly'
    print('TurboQuantFlashAttention prefill PASSED (matches DefaultFlashAttention).')

    # decode: passes quantised uint8 cache to flash_attn_with_kvcache
    # NOTE: The fused TurboQuant decode kernel is not yet implemented.
    # flash_attn_with_kvcache will reject uint8 cache or return garbage.
    # This test documents current behaviour rather than asserting correctness.
    print('\nTurboQuant decode (known limitation — fused kernel not yet implemented):')
    try:
        out_tq_dc = tqfa.decode(
            q_dc, k_cq, v_cq,
            scale=SCALE,
            cache_seqlens=cache_seqlens,
            block_table=block_tables,
        )
        print(f'  decode ran — output shape {out_tq_dc.shape} (results NOT meaningful with uint8 cache)')
    except Exception as e:
        print(f'  decode raised {type(e).__name__}: {e}')
        print('  (expected — standard flash_attn does not support uint8 cache)')

## 5. Attention Module (GPU)

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    import torch
    from nanovllm.layers.attention import Attention
    from nanovllm.kvcache.default import DefaultKVCache
    from nanovllm.layers.default_flash_attn import DefaultFlashAttention
    from nanovllm.utils.context import set_context, reset_context

    NH, NKV, HD = 8, 4, 64
    SCALE = HD ** -0.5
    NB, BK = 16, 16

    # Build the attention module
    cache_backend = DefaultKVCache(
        num_blocks=NB, block_size=BK,
        num_heads=NKV, head_dim=HD, dtype=torch.float16,
    )
    attn_backend = DefaultFlashAttention()

    attn = Attention(NH, HD, SCALE, NKV,
                     cache_backend=cache_backend,
                     attn_backend=attn_backend)

    k_c2, v_c2 = cache_backend.allocate()
    attn.k_cache = k_c2
    attn.v_cache = v_c2
    attn.additional_cache_tensors = ()

    # prefill forward
    T = 20
    q = torch.randn(T, NH,  HD, dtype=torch.float16, device='cuda')
    k = torch.randn(T, NKV, HD, dtype=torch.float16, device='cuda')
    v = torch.randn(T, NKV, HD, dtype=torch.float16, device='cuda')

    slot_map = torch.arange(T, dtype=torch.int32, device='cuda')
    cu_q = torch.tensor([0, T], dtype=torch.int32, device='cuda')
    cu_k = cu_q.clone()

    set_context(True, cu_q, cu_k, T, T, slot_map, None, None)
    out_pf2 = attn(q, k, v)
    reset_context()

    assert out_pf2.shape == (T, NH, HD)
    assert not out_pf2.isnan().any()
    torch.cuda.synchronize()
    print(f'Attention prefill: {out_pf2.shape}  PASSED.')

    # decode forward
    BS = 2
    q_d = torch.randn(BS, NH, HD, dtype=torch.float16, device='cuda')
    k_d = torch.randn(BS, NKV, HD, dtype=torch.float16, device='cuda')
    v_d = torch.randn(BS, NKV, HD, dtype=torch.float16, device='cuda')

    slot_d = torch.tensor([0, BK], dtype=torch.int32, device='cuda')  # first slot of block 0, 1
    ctx_lens = torch.tensor([5, 8], dtype=torch.int32, device='cuda')
    # block table: seq 0 → block 0, seq 1 → block 1
    btable = torch.tensor([[0, -1], [1, -1]], dtype=torch.int32, device='cuda')

    set_context(False, slot_mapping=slot_d, context_lens=ctx_lens, block_tables=btable)
    out_dc2 = attn(q_d, k_d, v_d)
    reset_context()

    assert out_dc2.shape[0] == BS
    assert not out_dc2.isnan().any()
    torch.cuda.synchronize()
    print(f'Attention decode : {out_dc2.shape}  PASSED.')

## 6. Attention with INT8 KV Cache (GPU)

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    from nanovllm.kvcache.int8 import Int8KVCache

    i8_be = Int8KVCache(
        num_blocks=NB, block_size=BK,
        num_heads=NKV, head_dim=HD, dtype=torch.float16,
    )
    k_c8, v_c8, k_sc, v_sc = i8_be.allocate()

    attn_i8 = Attention(NH, HD, SCALE, NKV,
                        cache_backend=i8_be,
                        attn_backend=DefaultFlashAttention())
    attn_i8.k_cache = k_c8
    attn_i8.v_cache = v_c8
    attn_i8.additional_cache_tensors = (k_sc, v_sc)

    # prefill — stores into INT8 cache
    set_context(True, cu_q, cu_k, T, T, slot_map, None, None)
    out_i8_pf = attn_i8(q, k, v)
    reset_context()
    assert not out_i8_pf.isnan().any()

    # The INT8-cached output will differ slightly from FP16 due to quantisation
    # (they're not identical because the K/V written to cache are quantised)
    print(f'INT8 Attention prefill output: {out_i8_pf.shape}  PASSED.')

    # decode — reads dequantised values from INT8 cache
    # First write k_d / v_d into the cache at decode slots
    i8_be.store(k_d, v_d, k_c8, v_c8, slot_d, k_sc, v_sc)
    torch.cuda.synchronize()

    # Manually dequantise and run attention to check pipeline works
    rk8, rv8 = i8_be.retrieve(k_c8, v_c8, k_sc, v_sc)
    assert rk8.dtype == torch.float16
    print(f'INT8 decode cache dequantised: {rk8.shape}  PASSED.')

## 7. TurboQuant Store Kernel: Numerical Sanity Check (GPU)

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    import math

    # Verify that k_norms are close to actual L2 norms of the stored keys
    expected_norms = key.float().norm(dim=-1)   # (N_TOKENS, NUM_HEADS)

    stored_norms = k_norms.view(-1)[:N_TOKENS * NUM_HEADS]
    stored_norms = stored_norms.view(N_TOKENS, NUM_HEADS).float()

    # Allow a small FP16 rounding error
    rel_err = (expected_norms - stored_norms).abs() / expected_norms.clamp(min=1e-6)
    max_err = rel_err.max().item()
    mean_err = rel_err.mean().item()
    print(f'k_norms  max rel error : {max_err:.4f}')
    print(f'k_norms  mean rel error: {mean_err:.4f}')
    assert max_err < 0.01, f'k_norm error too large: {max_err}'

    # Verify v_zeros ≈ actual per-vector min of value
    expected_zeros = value.float().min(dim=-1).values   # (N_TOKENS, NUM_HEADS)
    stored_zeros = v_zeros.view(-1)[:N_TOKENS * NUM_HEADS].view(N_TOKENS, NUM_HEADS).float()
    min_err = (expected_zeros - stored_zeros).abs().max().item()
    print(f'v_zeros  max abs error : {min_err:.5f}')
    assert min_err < 0.01

    print('TurboQuant store kernel numerical sanity PASSED.')

## 8. BlockManager: Decode Loop with Slot Verification (GPU)

In [ ]:
if not CUDA:
    print('Skipped.')
else:
    # Simulate what model_runner.prepare_decode does and verify slots
    # are always valid indices into the cache.

    BLOCK_SZ2 = 16
    Sequence.block_size = BLOCK_SZ2
    bm = BlockManager(64, BLOCK_SZ2)

    # Test several starting prompt lengths including multiples of block_size
    for start_len in [7, 8, 16, 17, 32, 33]:
        s = make_seq(start_len)
        bm2 = BlockManager(64, BLOCK_SZ2)
        bm2.allocate(s)

        max_slot = bm2.block_size * len(bm2.used_block_ids)

        for step in range(32):
            assert bm2.can_append(s), f'start={start_len} step={step}: cannot append'
            bm2.may_append(s)

            # compute slot exactly as prepare_decode does
            slot = s.block_table[-1] * bm2.block_size + s.last_block_num_tokens - 1

            # slot must be within the allocated cache space
            n_phys = len(bm2.used_block_ids) * bm2.block_size
            assert 0 <= slot < n_phys, (
                f'start={start_len} step={step}: slot={slot} out of range [0, {n_phys})'
            )
            s.append_token(step)

        bm2.deallocate(s)
        print(f'  start_len={start_len:2d}: slot verification across 32 decode steps PASSED.')

    print('BlockManager decode slot verification PASSED for all prompt lengths.')

## 9. Summary

In [ ]:
print('='*60)
print('TEST SUMMARY')
print('='*60)
tests = [
    'Registry (model / kvcache / flash-attn)',
    'Sequence properties and append_token',
    'BlockManager allocate / deallocate',
    'BlockManager exact-block-size edge case (bug-fix)',
    'BlockManager may_append across block boundaries',
    'Scheduler prefill → decode transition',
    'DefaultKVCache store + retrieve',
    'Int8KVCache store + round-trip accuracy',
    'TurboQuantKVCache allocate + store kernel',
    'DefaultFlashAttention prefill + decode',
    'TurboQuantFlashAttention prefill (matches Default)',
    'Attention module prefill + decode (Default backend)',
    'Attention module prefill (INT8 backend)',
    'TurboQuant store kernel numerical sanity',
    'BlockManager decode slot verification (all prompt lengths)',
]
for t in tests:
    print(f'  PASS  {t}')
print('='*60)